In [ ]:
# -*- coding: utf-8 -*-
"""geneformer_finetune_real_data.ipynb

# Geneformer Pretrained Fine-Tuning Benchmark — Phase 8

**What this notebook tests:**
Fine-tuning the *pretrained* Geneformer transformer (Genecorpus-104M, 6-layer, 10.6M param BERT)
on real datasets (80/20 train/test split). Instead of simulated scenarios with replicates,
this notebook iterates over all independent `.h5ad` samples in the `h5ad_real` directory.

**GPU:** L4 (22.5 GB VRAM).
**Crash safety:** Each run saves immediately to Drive. Re-running skips completed runs.
"""

from google.colab import drive
drive.mount('/content/drive')

!pip install -q transformers scanpy anndata scikit-learn

In [ ]:
import os
import glob

# ── Allocator config — reduces fragmentation, set before any CUDA ops ─────
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# ── EDIT THESE TO MATCH YOUR DRIVE ────────────────────────────────────────
DRIVE_ROOT      = "/content/drive/MyDrive/Thesis FM Computing"
H5AD_DIR        = f"{DRIVE_ROOT}/h5ad_real"
RESULTS_DIR     = f"{DRIVE_ROOT}/phase8_results"
CKPT_DIR        = f"{DRIVE_ROOT}/phase8_checkpoints/geneformer_finetune"

# HF Hub identifier for the pretrained model
PRETRAINED_MODEL     = "ctheodoris/Geneformer"
# Pin a specific official build. "Geneformer-V2-104M" is the standard 12-layer V2 model (104M-cell corpus)
# and pairs with the 104M token/median dictionaries loaded below. The bare repo default is a MOVING target
# (currently a ~316M model), so we pin the subfolder for reproducibility.
# Other builds in the repo: "Geneformer-V2-316M" (20-layer), "Geneformer-V1-10M".
GENEFORMER_SUBFOLDER = "Geneformer-V2-104M"
# Set to a commit SHA from huggingface.co/ctheodoris/Geneformer/commits/main to fully pin; None = latest.
GENEFORMER_REVISION  = None
# ─────────────────────────────────────────────────────────────────────────

MODEL_NAME = "Geneformer_pretrained"
TRAIN_SEED = 42

# Fine-tuning hyperparameters
BATCH_SIZE   = 2
ACCUM_STEPS  = 4
MAX_EPOCHS   = 30
LR           = 1e-4
PATIENCE     = 5
WEIGHT_DECAY = 1e-4

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CKPT_DIR,    exist_ok=True)

# Dynamically find all samples in the directory
sample_paths = sorted(glob.glob(os.path.join(H5AD_DIR, "*.h5ad")))
# Remove any intermediate mapped caches if they exist
sample_paths =[p for p in sample_paths if "_geneformer" not in p]
SAMPLES =[os.path.basename(p).replace(".h5ad", "") for p in sample_paths]

print(f"Found {len(SAMPLES)} samples in {H5AD_DIR}")
print(f"RESULTS_DIR : {RESULTS_DIR}")
print(f"CKPT_DIR    : {CKPT_DIR}")

In [ ]:
import sys, time, warnings, random, gc, pickle, urllib.request
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
from scipy.sparse import issparse

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from sklearn.model_selection import train_test_split
from sklearn.metrics import (cohen_kappa_score, f1_score,
                              matthews_corrcoef, accuracy_score)

import transformers
print(f"transformers version: {transformers.__version__}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
if DEVICE.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU    : {props.name}")
    print(f"VRAM   : {props.total_memory / 1e9:.1f} GB")

In [ ]:
# ── Geneformer Dictionaries & Ensembl Mapping ────────────────────────────────

TOKEN_DICT_PATH = "/tmp/token_dictionary.pkl"
MEDIAN_DICT_PATH = "/tmp/gene_median_dictionary.pkl"
HGNC_MAPPING_PATH = "/tmp/hgnc_to_ensembl.csv"

def get_geneformer_dicts():
    # The 104M (V2) dictionaries live INSIDE the model repo's geneformer/ package folder, and pair
    # with the pinned Geneformer-V2-104M build. (The old datasets/theodoris-lab/Genecorpus-104M path 404s.)
    TOKEN_DICT_URL  = "https://huggingface.co/ctheodoris/Geneformer/resolve/main/geneformer/token_dictionary_gc104M.pkl"
    MEDIAN_DICT_URL = "https://huggingface.co/ctheodoris/Geneformer/resolve/main/geneformer/gene_median_dictionary_gc104M.pkl"

    try:
        if not os.path.exists(TOKEN_DICT_PATH):
            print("Downloading Geneformer 104M token dictionary...")
            urllib.request.urlretrieve(TOKEN_DICT_URL, TOKEN_DICT_PATH)
        if not os.path.exists(MEDIAN_DICT_PATH):
            print("Downloading Geneformer 104M median dictionary...")
            urllib.request.urlretrieve(MEDIAN_DICT_URL, MEDIAN_DICT_PATH)
    except Exception as e:
        raise RuntimeError(f"Failed to fetch Geneformer dictionaries from Hugging Face: {e}")

    with open(TOKEN_DICT_PATH, "rb") as f:
        token_dictionary = pickle.load(f)
    with open(MEDIAN_DICT_PATH, "rb") as f:
        gene_median_dictionary = pickle.load(f)

    special_tokens = ["<pad>", "<mask>", "<unk>"]
    genes = sorted([k for k in token_dictionary.keys() if k not in special_tokens])
    print(f"Loaded Geneformer token dictionary: {len(genes)} Ensembl IDs available.")
    return token_dictionary, gene_median_dictionary, genes

def get_hgnc_to_ensembl_mapping():
    """
    Geneformer strictly requires Ensembl IDs. Real datasets often use HGNC symbols.
    This fetches the official HGNC mapping to safely translate gene symbols to Ensembl.
    """
    HGNC_URL = "https://storage.googleapis.com/public-download-files/hgnc/tsv/tsv/hgnc_complete_set.txt"
    if not os.path.exists(HGNC_MAPPING_PATH):
        print("Downloading HGNC to Ensembl mapping database...")
        urllib.request.urlretrieve(HGNC_URL, HGNC_MAPPING_PATH)

    df = pd.read_csv(HGNC_MAPPING_PATH, sep="\t", low_memory=False,
                     usecols=["symbol", "ensembl_gene_id", "alias_symbol", "prev_symbol"])
    df = df.dropna(subset=["ensembl_gene_id"])
    mapping = dict(zip(df["symbol"], df["ensembl_gene_id"]))
    # Also resolve aliases and previous symbols so renamed genes still map (primary symbol takes priority).
    for col in ("alias_symbol", "prev_symbol"):
        for aliases, ens in zip(df[col].fillna(""), df["ensembl_gene_id"]):
            for a in str(aliases).split("|"):
                a = a.strip()
                if a and a not in mapping:
                    mapping[a] = ens
    print(f"Loaded HGNC-to-Ensembl mapping ({len(mapping)} symbols incl. aliases/prev).")
    return mapping

TOKEN_DICT, MEDIAN_DICT, GENE_PANEL = get_geneformer_dicts()
SYMBOL_TO_ENSEMBL = get_hgnc_to_ensembl_mapping()

In [ ]:
def ensure_raw_counts(adata):
    # Raw integer counts are stored in adata.layers['counts']; adata.X is z-scored (has negatives).
    # Prefer the counts layer, then .raw, then fall back to X (guarded).
    if "counts" in adata.layers:
        X, var = adata.layers["counts"], adata.var
    elif adata.raw is not None:
        X, var = adata.raw.X, adata.raw.var
    else:
        X, var = adata.X, adata.var
    if issparse(X):
        X = X.toarray()
    X = np.asarray(X, dtype=np.float32)
    if X.min() < 0:
        raise ValueError(
            "Count matrix has negative values \u2014 layers['counts'] is not raw counts "
            "(adata.X is z-scored). Cannot tokenize."
        )
    return ad.AnnData(
        X   = np.round(X).astype(np.float32),
        obs = adata.obs.copy(),
        var = var.copy(),
        uns = adata.uns.copy()
    )

def load_sample(sample_name):
    """Load real data sample and align gene names for Geneformer."""
    raw_path = os.path.join(H5AD_DIR, f"{sample_name}.h5ad")

    if not os.path.exists(raw_path):
        raise FileNotFoundError(f"Cannot find: {raw_path}")

    adata = sc.read_h5ad(raw_path)
    adata = ensure_raw_counts(adata)

    # Geneformer's vocab is keyed on UNVERSIONED Ensembl gene IDs. Datasets may carry HGNC symbols
    # or versioned Ensembl IDs (ENSG00000123456.7) -> normalise both.
    raw_names = [str(g) for g in adata.var_names]
    first_gene = raw_names[0]
    if first_gene.startswith("ENSG"):
        new_names = [g.split(".")[0] for g in raw_names]   # strip version suffix
    else:
        print(f"  [INFO] Dataset genes look like symbols (e.g., '{first_gene}'). Converting to Ensembl...")
        new_names = [SYMBOL_TO_ENSEMBL.get(g, g).split(".")[0] for g in raw_names]

    converted = sum(1 for g in new_names if g.startswith("ENSG"))
    pct = (converted / len(new_names)) * 100
    print(f"  [INFO] {converted}/{len(new_names)} genes ({pct:.1f}%) resolved to Ensembl IDs.")
    if pct < 40:
        print(f"  [WARNING] Only {pct:.1f}% of genes mapped to Ensembl -> Geneformer input will be "
              f"sparse/unreliable for this sample. Check the dataset's gene identifiers.")
    adata.var_names = new_names
    adata.var_names_make_unique()

    # Rename 'Ground_Truth_Celltype' to 'cell_type' if it exists
    if "Ground_Truth_Celltype" in adata.obs.columns:
        adata.obs["cell_type"] = adata.obs["Ground_Truth_Celltype"]
        print("  [INFO] Renamed 'Ground_Truth_Celltype' to 'cell_type'.")

    assert "cell_type" in adata.obs.columns, (
        f"'cell_type' column missing. Available: {list(adata.obs.columns)}"
    )

    # Drop cells with NaN cell types
    if adata.obs["cell_type"].isna().any():
        n_nan = adata.obs["cell_type"].isna().sum()
        print(f"  [WARN] Dropping {n_nan} cells with NaN in 'cell_type'")
        adata = adata[~adata.obs["cell_type"].isna()].copy()

    n_types = adata.obs["cell_type"].nunique()
    print(f"  Loaded {sample_name}: {adata.shape[0]} cells × {adata.shape[1]} genes | {n_types} cell types")
    return adata

def stratified_split(adata, test_size=0.2):
    counts = adata.obs["cell_type"].value_counts()
    singletons = counts[counts < 2].index.tolist()
    if singletons:
        print(f"  [FIX] Removing singleton classes: {singletons}")
        adata = adata[~adata.obs["cell_type"].isin(singletons)].copy()

    labels = adata.obs["cell_type"].values
    idx    = np.arange(len(labels))
    tr_idx, te_idx = train_test_split(idx, test_size=test_size, stratify=labels, random_state=TRAIN_SEED)
    return adata[tr_idx].copy(), adata[te_idx].copy()

def encode_labels(train_a, test_a):
    all_types = sorted(train_a.obs["cell_type"].unique())
    label2id  = {t: i for i, t in enumerate(all_types)}
    id2label  = {i: t for t, i in label2id.items()}
    y_train   = np.array([label2id[t] for t in train_a.obs["cell_type"]])
    y_test    = np.array([label2id[t] for t in test_a.obs["cell_type"]])
    return label2id, id2label, y_train, y_test

def to_dense(adata):
    X = adata.X
    if issparse(X): X = X.toarray()
    return X.astype(np.float32)

In [ ]:
def tokenise_for_geneformer(X_raw, gene_names, token_dictionary, median_dictionary):
    """
    Geneformer Rank Value Encoding pipeline.
    1. Normalize expression by library size (n_counts) -> CP10k
    2. Divide by Genecorpus-104M non-zero median.
    3. Sort non-zero genes descending by this normalized rank.
    4. Map to token IDs and truncate to max length 2048.
    """
    n_cells, n_genes = X_raw.shape
    max_len = 2048

    # Map gene_names (Ensembl IDs) to their token IDs and medians
    gene_token_ids = np.zeros(n_genes, dtype=np.int64)
    gene_medians = np.ones(n_genes, dtype=np.float32)

    mapped_count = 0
    for i, g in enumerate(gene_names):
        if g in token_dictionary:
            gene_token_ids[i] = token_dictionary[g]
            gene_medians[i] = median_dictionary.get(g, 1.0)
            mapped_count += 1

    pct_mapped = (mapped_count / n_genes) * 100
    print(f"  Matched to Geneformer vocab: {mapped_count}/{n_genes} genes ({pct_mapped:.1f}%)")

    n_counts = X_raw.sum(axis=1, keepdims=True)
    n_counts = np.where(n_counts == 0, 1.0, n_counts) # avoid division by zero

    # Normalize by library size (standard Geneformer factor is 10,000)
    X_norm = (X_raw / n_counts) * 10_000.0

    # Normalize by median dataset expression
    X_scaled = X_norm / gene_medians

    pad_token_id = token_dictionary.get("<pad>", 0)
    input_ids = np.full((n_cells, max_len), pad_token_id, dtype=np.int64)
    attention_mask = np.zeros((n_cells, max_len), dtype=np.int64)

    for i in range(n_cells):
        row = X_scaled[i]
        nonzero_idx = np.nonzero(row)[0]

        # Only keep genes that actually have a valid token mapping
        valid_nonzero = [idx for idx in nonzero_idx if gene_token_ids[idx] != 0]
        if len(valid_nonzero) == 0:
            continue

        valid_nonzero = np.array(valid_nonzero)

        # Sort nonzero indices by scaled expression (descending)
        sorted_nonzero = valid_nonzero[np.argsort(row[valid_nonzero])[::-1]]

        # Truncate to maximum sequence length
        sorted_nonzero = sorted_nonzero[:max_len]

        # Map to valid token IDs
        tokens = gene_token_ids[sorted_nonzero]
        seq_len = len(tokens)

        input_ids[i, :seq_len] = tokens
        attention_mask[i, :seq_len] = 1

    tensor_mb = (input_ids.nbytes + attention_mask.nbytes) / 1024**2
    print(f"  Tokenised {n_cells} cells | tensor size: {tensor_mb:.1f} MB")

    return torch.tensor(input_ids, dtype=torch.long), torch.tensor(attention_mask, dtype=torch.long)

In [ ]:
class GeneformerPretrained(nn.Module):
    """
    Geneformer architecture initialized with pretrained weights from Genecorpus 104M.
    We use HuggingFace's `from_pretrained` which downloads the exact model weights,
    but dynamically replaces the classification head for our specific number of classes.
    """
    def __init__(self, num_classes, pad_token_id=0):
        super().__init__()
        from transformers import BertForSequenceClassification
        import logging

        # Suppress HF warnings about randomly initializing the classifier head
        logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)

        _rev = f" @ {GENEFORMER_REVISION}" if GENEFORMER_REVISION else ""
        print(f"  Loading pretrained Geneformer ({PRETRAINED_MODEL}/{GENEFORMER_SUBFOLDER}{_rev})...")
        self.model = BertForSequenceClassification.from_pretrained(
            PRETRAINED_MODEL,
            subfolder=GENEFORMER_SUBFOLDER,   # pin a specific official build (paired with the 104M dicts)
            revision=GENEFORMER_REVISION,     # pin repo commit for reproducibility (None = latest)
            num_labels=num_classes,
            ignore_mismatched_sizes=True,     # only the classifier head differs in size
            pad_token_id=pad_token_id
        )

        # Only resize if the checkpoint vocab differs from the paired token dictionary. With the correctly
        # paired V2-104M build + 104M dicts this should be a no-op; kept as a guard, not a blind inflation.
        if self.model.config.vocab_size != len(TOKEN_DICT):
            print(f"  [INFO] Resizing embeddings {self.model.config.vocab_size} -> {len(TOKEN_DICT)} "
                  f"to match paired token dictionary.")
            self.model.resize_token_embeddings(len(TOKEN_DICT))

        n_params = sum(p.numel() for p in self.parameters())
        print(f"  Geneformer: {num_classes} classes, {n_params:,} params — Pretrained init")

    def forward(self, input_ids, attention_mask):
        # NOTE: BertForSequenceClassification extracts the hidden state of
        # the first token. In Geneformer, without a <CLS> token, the first
        # token (the highest-expressed gene) acts natively as the contextual cell
        # representation by having absorbed sequence-wide info via self-attention.
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
        return outputs.logits

In [ ]:
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total, n = 0.0, 0
    optimizer.zero_grad()

    for step, (ids, mask, yb) in enumerate(loader):
        ids, mask, yb = ids.to(DEVICE), mask.to(DEVICE), yb.to(DEVICE)

        with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
            logits = model(ids, mask)
            loss = criterion(logits.float(), yb) / ACCUM_STEPS

        scaler.scale(loss).backward()

        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        total += loss.item() * ACCUM_STEPS * len(yb)
        n += len(yb)

    return total / n

@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    total, n, preds = 0.0, 0,[]
    for ids, mask, yb in loader:
        ids, mask, yb = ids.to(DEVICE), mask.to(DEVICE), yb.to(DEVICE)
        with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
            logits = model(ids, mask)
        total += criterion(logits.float(), yb).item() * len(yb); n += len(yb)
        preds.extend(logits.argmax(1).cpu().numpy())
    return total / n, np.array(preds)

def fit_model(model, X_tr_ids, X_tr_mask, y_tr, X_te_ids, X_te_mask, y_te, sample_name):
    tr_dl = DataLoader(TensorDataset(X_tr_ids, X_tr_mask, torch.tensor(y_tr, dtype=torch.long)),
                       batch_size=BATCH_SIZE, shuffle=True, pin_memory=(DEVICE.type == "cuda"))
    te_dl = DataLoader(TensorDataset(X_te_ids, X_te_mask, torch.tensor(y_te, dtype=torch.long)),
                       batch_size=BATCH_SIZE, shuffle=False, pin_memory=(DEVICE.type == "cuda"))

    # Compute inverse-frequency class weights
    w = 1.0 / (np.bincount(y_tr) + 1e-6)
    w = w / w.sum() * len(w)
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32).to(DEVICE))

    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
    ckpt = os.path.join(CKPT_DIR, f"geneformer_finetune_{sample_name}_best.pt")

    best_val, pat = float("inf"), 0
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()

    for epoch in range(1, MAX_EPOCHS + 1):
        tr_loss = train_epoch(model, tr_dl, optimizer, criterion)
        vl_loss, _ = eval_epoch(model, te_dl, criterion)
        scheduler.step()

        if vl_loss < best_val:
            best_val, pat = vl_loss, 0
            torch.save(model.state_dict(), ckpt)
        else:
            pat += 1

        if epoch % 5 == 0:
            print(f"    Epoch {epoch:02d} | train={tr_loss:.4f} val={vl_loss:.4f} | patience={pat}/{PATIENCE}")

        if pat >= PATIENCE:
            print(f"    Early stop at epoch {epoch}")
            break

    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    _, final_preds = eval_epoch(model, te_dl, criterion)

    runtime = time.time() - t0
    peak_mb = torch.cuda.max_memory_allocated() / 1024**2 if DEVICE.type == "cuda" else 0
    print(f"    Done in {runtime/60:.1f} min | peak GPU: {peak_mb:.0f} MB")
    return final_preds, runtime, peak_mb

In [ ]:
def compute_metrics(y_true, y_pred, id2label):
    kappa    = cohen_kappa_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    mcc      = matthews_corrcoef(y_true, y_pred)
    acc      = accuracy_score(y_true, y_pred)

    unique, counts = np.unique(y_true, return_counts=True)
    rarest = unique[np.argmin(counts)]
    rare_f1 = f1_score((y_true == rarest).astype(int),
                       (y_pred == rarest).astype(int), zero_division=0)
    return {
        "kappa":          round(float(kappa),    4),
        "macro_f1":       round(float(macro_f1), 4),
        "mcc":            round(float(mcc),      4),
        "accuracy":       round(float(acc),      4),
        "rare_type_f1":   round(float(rare_f1),  4),
        "rare_type_name": id2label[rarest],
        "rare_type_n":    int(counts.min()),
        "n_test":         int(len(y_true)),
        "n_classes":      int(len(id2label)),
    }

def print_metrics(m):
    print(f"  κ (kappa)    : {m['kappa']:.4f}")
    print(f"  Macro F1     : {m['macro_f1']:.4f}")
    print(f"  MCC          : {m['mcc']:.4f}")
    print(f"  Accuracy     : {m['accuracy']:.4f}")
    print(f"  Rare type F1 : {m['rare_type_f1']:.4f}  ({m['rare_type_name']}, n_test={m['rare_type_n']})")

In [ ]:
CSV_PATH = os.path.join(RESULTS_DIR, "phase8_results_Geneformer.csv")

def save_result(sample_name, metrics, predictions, runtime_s, peak_gpu_mb):
    row = {
        "model":       MODEL_NAME,
        "sample":      sample_name,
        "condition":   "pretrained_finetune",
        "runtime_s":   round(runtime_s, 1),
        "peak_gpu_mb": round(peak_gpu_mb, 0),
        **metrics
    }
    row_df = pd.DataFrame([row])
    if os.path.exists(CSV_PATH):
        row_df.to_csv(CSV_PATH, mode="a", header=False, index=False)
    else:
        row_df.to_csv(CSV_PATH, mode="w", header=True,  index=False)

    pred_fname = f"{MODEL_NAME}_{sample_name}_preds.npy"
    np.save(os.path.join(RESULTS_DIR, pred_fname), predictions)
    print(f"  Saved metrics → phase8_results_Geneformer.csv")
    print(f"  Saved preds   → {pred_fname}")

def check_done(sample_name):
    if not os.path.exists(CSV_PATH): return False
    df = pd.read_csv(CSV_PATH)
    return ((df["model"]  == MODEL_NAME)  &
            (df["sample"] == sample_name) &
            (df["condition"] == "pretrained_finetune")).any()

In [ ]:
completed, total = 0, len(SAMPLES)

print("=" * 65)
print(f"Geneformer Pretrained Fine-Tuning | {total} real samples")
print("=" * 65)

for sample in SAMPLES:
    print(f"\n{'─'*65}")
    print(f"[{completed+1}/{total}]  Sample: {sample}")
    print(f"{'─'*65}")

    if DEVICE.type == "cuda":
        free_gb, total_gb =[x/1e9 for x in torch.cuda.mem_get_info()]
        print(f"  GPU memory: {free_gb:.1f} GB free / {total_gb:.1f} GB total")

    if check_done(sample):
        print(f"  [SKIP] Already in results CSV")
        completed += 1
        continue

    # ── 1. Load data ──────────────────────────────────────────────────────
    try:
        adata = load_sample(sample)
    except FileNotFoundError as e:
        print(f"  [WARN] {e}\n  Skipping sample.")
        continue

    # ── 2. Split ──────────────────────────────────────────────────────────
    train_a, test_a = stratified_split(adata)
    label2id, id2label, y_train, y_test = encode_labels(train_a, test_a)
    n_classes = len(label2id)
    print(f"  Train: {len(y_train)} cells | Test: {len(y_test)} cells | Classes: {n_classes}")

    # ── 3. Tokenise ───────────────────────────────────────────────────────
    print("  Tokenising...")
    gene_names = list(adata.var_names)

    X_train_dense = to_dense(train_a)
    del train_a; gc.collect()
    X_tr_ids, X_tr_mask = tokenise_for_geneformer(X_train_dense, gene_names, TOKEN_DICT, MEDIAN_DICT)
    del X_train_dense; gc.collect()

    X_test_dense = to_dense(test_a)
    del adata, test_a; gc.collect()
    X_te_ids, X_te_mask = tokenise_for_geneformer(X_test_dense, gene_names, TOKEN_DICT, MEDIAN_DICT)
    del X_test_dense; gc.collect()

    # ── 4. Build Pretrained model ─────────────────────────────────────────
    PAD_TOKEN_ID = TOKEN_DICT.get("<pad>", 0)

    model = GeneformerPretrained(
        num_classes=n_classes,
        pad_token_id=PAD_TOKEN_ID
    ).to(DEVICE)

    # ── 5. Train ──────────────────────────────────────────────────────────
    print("  Training...")
    preds, runtime, peak_mb = fit_model(
        model, X_tr_ids, X_tr_mask, y_train, X_te_ids, X_te_mask, y_test, sample
    )

    # ── 6. Evaluate ───────────────────────────────────────────────────────
    metrics = compute_metrics(y_test, preds, id2label)
    print("  Results:")
    print_metrics(metrics)

    # ── 7. Save ───────────────────────────────────────────────────────────
    save_result(sample, metrics, preds, runtime, peak_mb)
    completed += 1

    # ── 8. Clean up GPU memory ────────────────────────────────────────────
    del model, X_tr_ids, X_tr_mask, X_te_ids, X_te_mask
    torch.cuda.synchronize()
    torch.cuda.empty_cache()
    gc.collect()

print("\n" + "=" * 65)
print(f"Complete: {completed}/{total} runs")
print("=" * 65)

In [ ]:
if os.path.exists(CSV_PATH):
    df   = pd.read_csv(CSV_PATH)
    mine = df[df["model"] == MODEL_NAME].copy()

    if len(mine) == 0:
        print("No results yet — run the main loop first.")
    else:
        print("=== Per-run results ===")
        cols =["sample", "kappa", "macro_f1", "rare_type_f1", "accuracy", "runtime_s", "peak_gpu_mb"]
        print(mine[cols].sort_values("sample").to_string(index=False))

        print(f"\nTotal samples recorded: {len(mine)}/{total}")
        print(f"Total runtime         : {mine['runtime_s'].sum()/3600:.2f} hours")
        print(f"Mean κ (all samples)  : {mine['kappa'].mean():.4f}")
else:
    print(f"No CSV found at {CSV_PATH}")

In [ ]:
from google.colab import runtime
runtime.unassign()